# Train our own plate detector (optional)

Runs on a free GPU (Kaggle or Colab). The default detector is a third-party AGPL-3.0 model (docs/decisions.md D5, D17); a model trained here on openly licensed data can become our own `models-v1` release.

**Fix the model-selection rule before looking at any numbers:** pick the checkpoint with the best mAP@0.5 on the *validation* split, then report it once on the held-out *test* split from a **different source**.

In [ ]:
# 1. Install (training only; the edge worker never needs these)
%pip install -q ultralytics==8.3.* roboflow

## 2. Data
Use datasets whose licence allows training and redistribution of weights. Record each dataset's name, URL and licence here:

| Split | Source | Licence | Images |
|---|---|---|---|
| train/val | _fill in_ | _fill in_ | _fill in_ |
| test (different source) | _fill in_ | _fill in_ | _fill in_ |

In [ ]:
from ultralytics import YOLO

DATA_YAML = "data.yaml"  # train/val paths, 1 class: plate
model = YOLO("yolo11n.pt")
results = model.train(data=DATA_YAML, imgsz=640, epochs=100, patience=20, seed=0)
best = YOLO(results.save_dir / "weights" / "best.pt")

In [ ]:
# 3. Held-out test split (from a different source), measured once
metrics = best.val(data="test.yaml", split="test")
print(metrics.box.map50, metrics.box.map)

In [ ]:
# 4. Export for the edge worker (ONNX Runtime, no PyTorch at run time)
best.export(format="onnx", imgsz=640, dynamic=True, simplify=True)
# Then: attach the .onnx to a GitHub Release `models-v1`, and put its URL + SHA-256 in anpr/download_model.py